# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Praveen23-kk/FlyRank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1:** "Engagement and Visibility Move Together" (Finding #5). The paper claims high scroll/engagement yields a higher Health Score.
* **Methodology Question:** Since Health Score is explicitly defined in the methodology as including "scroll depth (20 pts)", doesn't using scroll metrics to predict or correlate with Health Score introduce target leakage? The label is derived directly from the feature being tested.

**Finding 2:** "The Freshness Multiplier" (Finding #4). The paper claims refreshing 365+ day content yields a 57x impression boost.
* **Methodology Question:** Does the validation design account for selection bias? Pages selected for a refresh after a year are likely historically valuable assets with proven demand, while the untouched baseline includes abandoned or low-value pages.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Load Data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df["high_visibility_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

# Define Features
leakage_cols = [
    'trend_direction', 'trend_pct', 'high_visibility_label', 'is_declining_label',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]
id_cols = ['content_id', 'client_id']
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features = [c for c in numeric_cols if c not in leakage_cols and c not in id_cols]
df[features] = df[features].fillna(0)

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(y_true)[order[:k]].mean())

# BEFORE: Random Split
X = df[features]
y = df['high_visibility_label']
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.3, random_state=42)

rf_rnd = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100)
rf_rnd.fit(X_train_rnd, y_train_rnd)
rnd_scores = rf_rnd.predict_proba(X_test_rnd)[:, 1]
p100_rnd = precision_at_k(y_test_rnd, rnd_scores, 100)

# AFTER: Honest Grouped Split (by client_id)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
X_train_grp, y_train_grp = train_df[features], train_df['high_visibility_label']
X_test_grp, y_test_grp = test_df[features], test_df['high_visibility_label']

rf_grp = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100)
rf_grp.fit(X_train_grp, y_train_grp)
grp_scores = rf_grp.predict_proba(X_test_grp)[:, 1]
p100_grp = precision_at_k(y_test_grp, grp_scores, 100)

print("===============================================================")
print("BEFORE/AFTER: RANDOM VS HONEST SPLIT (Precision@100)")
print("===============================================================")
print(f"Random Split (memorizes clients): {p100_rnd:.4f}")
print(f"Grouped Split (honest unseen clients): {p100_grp:.4f}")
print(f"Gap (Overestimation): {p100_rnd - p100_grp:.4f}")


BEFORE/AFTER: RANDOM VS HONEST SPLIT (Precision@100)
Random Split (memorizes clients): 0.8600
Grouped Split (honest unseen clients): 0.6800
Gap (Overestimation): 0.1800


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# Deliberately adding a leaky feature (target leakage)
leaky_features = features + ['trend_pct'] # trend_pct directly determines the label
df[leaky_features] = df[leaky_features].fillna(0)

# Train with leakage on honest split
X_train_leak, y_train_leak = train_df[leaky_features], train_df['high_visibility_label']
X_test_leak, y_test_leak = test_df[leaky_features], test_df['high_visibility_label']

rf_leak = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100)
rf_leak.fit(X_train_leak, y_train_leak)
leak_scores = rf_leak.predict_proba(X_test_leak)[:, 1]
p100_leak = precision_at_k(y_test_leak, leak_scores, 100)

print("===============================================================")
print("LEAKAGE AUDIT: DELIBERATE TARGET LEAKAGE")
print("===============================================================")
print(f"Honest Grouped Split (clean features): {p100_grp:.4f}")
print(f"Leaky Feature Added (trend_pct): {p100_leak:.4f}")
print("Conclusion: The harness correctly catches the leakage, jumping near 1.0.")


LEAKAGE AUDIT: DELIBERATE TARGET LEAKAGE
Honest Grouped Split (clean features): 0.6800
Leaky Feature Added (trend_pct): 1.0000
Conclusion: The harness correctly catches the leakage, jumping near 1.0.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Bold Claim:** "The model perfectly identifies which pages will drop in traffic by finding a correlation between lower engagement and decline."

**Rewritten Safe Claim:** "We observed a directional correlation between lower historical engagement and subsequent traffic declines, which allows the model to flag at-risk content for editorial review."


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.